<a href="https://colab.research.google.com/github/ZacharyMalonjao/BoxOfficeData/blob/main/BoxOfficeDataCleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from scipy import stats

In [ ]:
#Grab files form github/ check for duplicates

df =  pd.read_csv("https://raw.githubusercontent.com/ZacharyMalonjao/BoxOfficeData/refs/heads/main/datasets/dirty_data.csv.csv")
print(df.duplicated().sum())

0


In [ ]:
#I N T E R V A L S

print(df['Year'].isna().sum())
df['Year'] =  df['Year'].astype(str).str.extract(r'(\d{4})')
df['Year'] = pd.to_numeric(df["Year"], errors ='coerce')
df[(df['Year'] < 2000) |(df['Year']>2025) ]
print(df['Year'].isna().sum())
#No outliers, invalid formats and missing values

0
0


In [ ]:
# D I S C R E T E
#----------------------
#1. VOTE COUNT
#    Check formatting and missing values
df['Vote_Count'] = pd.to_numeric(df['Vote_Count'], errors = 'coerce')
old_outliers = df[
    (df['Vote_Count']<0)| (df['Vote_Count'] > df['Vote_Count'].quantile(0.99) )
]
print("Old VoteCount Outliers:", old_outliers.shape[0])
print("Old Missing VoteCount values", df['Vote_Count'].isna().sum())

#    Mean Imputation
votecount_mean =  df[df['Vote_Count'] >=0]['Vote_Count'].mean()
votecount_q99 =  df['Vote_Count'].quantile(0.99) #Freeze Threshold

df['Vote_Count'] = df['Vote_Count'].apply(
    lambda x: votecount_mean if (pd.isna(x) or x < 0 or x > votecount_q99) else x
)

print("Missing Vote Count values after cleaning:", df["Vote_Count"].isna().sum())
print("Outliers in vote count after cleaning:", df[
    (df["Vote_Count"] < 0) | (df["Vote_Count"] > votecount_q99)
].shape[0])

#2. Rank
#  Find missing values
missing = df['Rank'].isna().sum()
print("Missing values in 'Rank':", missing)
#  Find outliers
invalid = df[(df['Rank'] < 1) | (df['Rank'] > 200)]
print("Invalid rank values:")
print(invalid[['Year','Rank']])

#   Check if there are 200 unique ranks
rank_counts = df.groupby('Year')['Rank'].nunique()
if (rank_counts == 200).all():
    print("All years have exactly 200 unique ranks.")
else:
    print("Some years do NOT have 200 unique ranks:")
    print(rank_counts[rank_counts != 200])



Old VoteCount Outliers: 49
Old Missing VoteCount values 170
Missing Vote Count values after cleaning: 0
Outliers in vote count after cleaning: 0
Missing values in 'Rank': 0
Invalid rank values:
Empty DataFrame
Columns: [Year, Rank]
Index: []
All years have exactly 200 unique ranks.


In [ ]:
# C O N T I N U O U S
#Method to clean continuous dtypes, except Rating because it has additional formatting issues
def clean_continuous(df, col_name, outlier_pct=0.99):
    """
    Cleans a numeric continuous column by:
    1. Converting to numeric
    2. Identifying old outliers (<0 or > percentile)
    3. Imputing missing and outlier values with the mean
    4. Reporting counts before and after cleaning

    Parameters:
        df (DataFrame): Your dataset
        col_name (str): Column to clean
        outlier_pct (float): Upper percentile threshold for outliers (default 0.99)
    """
    # Convert to numeric
    df[col_name] = pd.to_numeric(df[col_name], errors='coerce')

    # Identify old outliers
    old_outliers = df[(df[col_name] < 0) | (df[col_name] > df[col_name].quantile(outlier_pct))]
    print(f" {col_name} Outliers Before cleaning:", old_outliers.shape[0])
    print(f"Missing {col_name} values before cleaning:", df[col_name].isna().sum())

    # Mean imputation & freeze upper percentile threshold
    col_mean = df[df[col_name] >= 0][col_name].mean()
    col_q = df[col_name].quantile(outlier_pct)

    # Replace missing or outlier values
    df[col_name] = df[col_name].apply(
        lambda x: col_mean if (pd.isna(x) or x < 0 or x > col_q) else x
    )

    # Report after cleaning
    print(f"Missing {col_name} values after cleaning:", df[col_name].isna().sum())
    print(f"Outliers in {col_name} after cleaning:",
          df[(df[col_name] < 0) | (df[col_name] > col_q)].shape[0])
    print("------------------------------------")
    return df

#Method to clean Rating because of formatting issues
def clean_rating(df, col_name = 'Rating'):

  """
  This method does the following
  1. Extract the number from the rating (e.g. 1.2 from 1.2/10)
  2. Validate(Check outliers and invalids)
  3. Impute Mean
  4. Report before and after
  """
  #Convert to numeric
  df["Rating_clean"] = df[col_name].str.extract(r'(\d+\.\d+)').astype(float)
  df["Rating_clean"] = pd.to_numeric(df["Rating_clean"], errors='coerce')
  #Outlier identification
  outliers = df[
      (df["Rating_clean"] <0) | (df["Rating_clean"] > 10)
  ]
  print("Ratings Outliers before:", outliers.shape[0])
  print("Ratings missings before: ", df["Rating_clean"].isna().sum())

  #Mean imputation
  col_mean = df[df["Rating_clean"] >=0]["Rating_clean"].mean()
  df["Rating_clean"] = df["Rating_clean"].apply(
    lambda x: col_mean if (pd.isna(x) or x < 0 or x > 10) else x
  )
  # After cleaning
  print("Ratings Outliers after:", df[(df["Rating_clean"] <0) | (df["Rating_clean"] > 10)].shape[0])
  print("Ratings missings after:", df["Rating_clean"].isna().sum())

  return df



#1. Wordlwide Gross
df = clean_continuous(df, 'Worldwide Gross')


#2. Domestic Gross
df = clean_continuous(df, 'Domestic Gross')

#3. Domestic %
df = clean_continuous(df, 'Domestic %')

#4. Foreign Gross

df = clean_continuous(df, 'Foreign Gross')
#5. Foreign %

df = clean_continuous(df, 'Foreign %')
#6. Rating
df = clean_rating(df)


 Worldwide Gross Outliers Before cleaning: 50
Missing Worldwide Gross values before cleaning: 0
Missing Worldwide Gross values after cleaning: 0
Outliers in Worldwide Gross after cleaning: 0
------------------------------------
 Domestic Gross Outliers Before cleaning: 50
Missing Domestic Gross values before cleaning: 0
Missing Domestic Gross values after cleaning: 0
Outliers in Domestic Gross after cleaning: 0
------------------------------------
 Domestic % Outliers Before cleaning: 49
Missing Domestic % values before cleaning: 0
Missing Domestic % values after cleaning: 0
Outliers in Domestic % after cleaning: 0
------------------------------------
 Foreign Gross Outliers Before cleaning: 50
Missing Foreign Gross values before cleaning: 0
Missing Foreign Gross values after cleaning: 0
Outliers in Foreign Gross after cleaning: 0
------------------------------------
 Foreign % Outliers Before cleaning: 0
Missing Foreign % values before cleaning: 0
Missing Foreign % values after cleani

In [ ]:
#N O M I N A L S

#1. Release Group
#  Edit spaces spaces
df['Release Group']= df['Release Group'] .astype(str).str.strip()
df['Release Group']= df['Release Group'] .str.replace(r"\s+", " ", regex=True)
#  normalize unicode
df['Release Group'] = df['Release Group'].str.normalize("NFKC")
#  report unusual entries
non_string = df[~df['Release Group'].apply(lambda x: isinstance(x,str))]
print("Non String entries in Release Group:", non_string.shape[0])
print("--------------------")

#2 and 3, Production COuntries and Genres have the same procedure so we
#can bundle them up in a single method

def clean_genre_and_production_countries( df, col_name):
 #   Standardize and take only the first primary genre or country
 df[col_name] =  df[col_name].str.split(",").str[0].str.strip()
 df[col_name] =  df[col_name].str.replace(r"\s+", " ", regex=True)
 df[col_name] =  df[col_name].str.normalize("NFKC")

 #  Modal Imputation
 print(f"Missing {col_name} before: ", df[col_name].isna().sum())
 mode_value= df[col_name].mode()[0]
 df[col_name] = df[col_name].fillna(mode_value)
 print(f"Missing {col_name} after: ", df[col_name].isna().sum())
 print("Mode:", mode_value)
 print("--------------------")



clean_genre_and_production_countries(df, 'Production_Countries')

clean_genre_and_production_countries(df, 'Genres')


#4. Original_Language
#  Raw Data has languages abbreviated, we need to create a dictionary
languages = {
    "en": "English",
    "zh": "Chinese",
    "ja": "Japanese",
    "fr": "French",
    "da": "Danish",
    "it": "Italian",
    "cn": "Chinese",
    "de": "German",
    "sr": "Serbian",
    "hi": "Hindi",
    "sv": "Swedish",
    "fa": "Persian (Farsi)",
    "es": "Spanish",
    "pt": "Portuguese",
    "ko": "Korean",
    "th": "Thai",
    "cs": "Czech",
    "el": "Greek",
    "tr": "Turkish",
    "ru": "Russian",
    "nl": "Dutch",
    "fi": "Finnish",
    "pl": "Polish",
    "no": "Norwegian",
    "tl": "Tagalog (Filipino)",
    "te": "Telugu",
    "ar": "Arabic",
    "ta": "Tamil",
    "id": "Indonesian",
    "vi": "Vietnamese",
    "ml": "Malayalam",
    "kn": "Kannada",
    "et": "Estonian",
    "pa": "Punjabi",
    "uk": "Ukrainian"
}
#  replace them according to the dictionary
df['Original_Language'] = df['Original_Language'].map(languages)
# Handle missings
missings =  df[df['Original_Language'].isna()]['Original_Language'].unique()

print("Missing languages before:", df["Original_Language"].isna().sum())
mode_val = df['Original_Language'].mode()[0]
df['Original_Language'] = df['Original_Language'].fillna(mode_val)
print("Missing languages after:", df["Original_Language"].isna().sum())





Non String entries in Release Group: 0
--------------------
Missing Production_Countries before:  200
Missing Production_Countries after:  0
Mode: United States of America
--------------------
Missing Genres before:  178
Missing Genres after:  0
Mode: Comedy
--------------------
Missing languages before: 170
Missing languages after: 0


In [ ]:
df.to_csv("clean_box_office_data.csv", index = False)
from google.colab import files
# files.download("clean_box_office_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>